# Day 8: FastAPI Advanced and Security
**Dependency injection, middleware, JWT authentication and PyMongo**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup
Colab uses mongomock (in-memory MongoDB). VS Code can use a real MongoDB by setting environment variables.

In [ ]:
# ===== Option A: Google Colab (active) =====
!pip install -q fastapi httpx pyjwt mongomock pymongo python-multipart
import os
os.environ["USE_MOCK"] = "1"

# ===== Option B: VS Code with real MongoDB (uncomment to use) =====
# pip install fastapi uvicorn httpx pyjwt pymongo python-multipart   (in your terminal)
# import os
# os.environ["USE_MOCK"] = "0"
# os.environ["MONGO_URI"] = "mongodb://localhost:27017"   # or your Atlas URI

os.environ.setdefault("SECRET_KEY", "dev-only-change-me")
print("Mock mode:", os.environ["USE_MOCK"])

## 2. Dependency injection basics

In [ ]:
from fastapi import FastAPI, Depends, Query
from fastapi.testclient import TestClient

demo = FastAPI()

def pagination(skip: int = Query(0, ge=0), limit: int = Query(10, ge=1, le=50)):
    return {"skip": skip, "limit": limit}

def get_resource():
    print("  setup: open resource")
    yield {"name": "fake-db"}
    print("  cleanup: close resource")

@demo.get("/items")
def items(page=Depends(pagination), res=Depends(get_resource)):
    return {"page": page, "resource": res["name"]}

c = TestClient(demo)
print(c.get("/items", params={"limit": 5}).json())
print("Invalid limit:", c.get("/items", params={"limit": 500}).status_code)

## 3. Middleware: timing and request ids

In [ ]:
import time, uuid

timed = FastAPI()     # a fresh app: middleware must be added before the app starts serving

@timed.get("/ping")
def ping():
    return {"pong": True}

@timed.middleware("http")
async def timing(request, call_next):
    start = time.perf_counter()
    response = await call_next(request)
    ms = (time.perf_counter() - start) * 1000
    response.headers["X-Process-Time-ms"] = f"{ms:.1f}"
    response.headers["X-Request-ID"] = str(uuid.uuid4())
    print(f"  {request.method} {request.url.path} -> {response.status_code} in {ms:.1f} ms")
    return response

r = TestClient(timed).get("/ping")
print(dict((k, v) for k, v in r.headers.items() if k.startswith("x-")))

## 4. Password hashing (built-in, no extra install)

In [ ]:
import hashlib, hmac, os

def hash_password(password: str) -> str:
    salt = os.urandom(16)
    digest = hashlib.pbkdf2_hmac("sha256", password.encode(), salt, 200_000)
    return salt.hex() + "$" + digest.hex()

def verify_password(password: str, stored: str) -> bool:
    salt_hex, digest_hex = stored.split("$")
    check = hashlib.pbkdf2_hmac("sha256", password.encode(), bytes.fromhex(salt_hex), 200_000)
    return hmac.compare_digest(check.hex(), digest_hex)

h1, h2 = hash_password("secret1"), hash_password("secret1")
print("Same password, different hashes:", h1[:20], "...", h2[:20], "...")
print(verify_password("secret1", h1), verify_password("wrong", h1))

## 5. JSON Web Tokens

In [ ]:
import jwt, datetime as dt
SECRET = os.environ["SECRET_KEY"]

def create_token(sub, role="member", minutes=30):
    exp = dt.datetime.now(dt.timezone.utc) + dt.timedelta(minutes=minutes)
    return jwt.encode({"sub": sub, "role": role, "exp": exp}, SECRET, algorithm="HS256")

token = create_token("hana")
print("Token:", token[:40], "...")
print("Decoded:", jwt.decode(token, SECRET, algorithms=["HS256"]))

expired = create_token("hana", minutes=-1)
try:
    jwt.decode(expired, SECRET, algorithms=["HS256"])
except jwt.ExpiredSignatureError:
    print("Expired token rejected")

try:
    jwt.decode(token, "wrong-key", algorithms=["HS256"])
except jwt.InvalidSignatureError:
    print("Forged token rejected")

## 6. The secure Notes API (FastAPI + JWT + MongoDB)

In [ ]:
%%writefile secure_notes.py
import datetime as dt, hashlib, hmac, os
import jwt
from bson import ObjectId
from fastapi import Depends, FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from pydantic import BaseModel, Field

SECRET = os.environ.get("SECRET_KEY", "dev-only-change-me")

if os.environ.get("USE_MOCK", "1") == "1":
    import mongomock
    client = mongomock.MongoClient()
else:
    from pymongo import MongoClient
    client = MongoClient(os.environ.get("MONGO_URI", "mongodb://localhost:27017"))
db = client["secure_notes"]

app = FastAPI(title="Secure Notes API")
app.add_middleware(CORSMiddleware, allow_origins=["http://localhost:3000"],
                   allow_methods=["*"], allow_headers=["*"])

# ---------- helpers ----------
def hash_password(pw):
    salt = os.urandom(16)
    return salt.hex() + "$" + hashlib.pbkdf2_hmac("sha256", pw.encode(), salt, 200_000).hex()

def verify_password(pw, stored):
    salt, digest = stored.split("$")
    check = hashlib.pbkdf2_hmac("sha256", pw.encode(), bytes.fromhex(salt), 200_000).hex()
    return hmac.compare_digest(check, digest)

def create_token(sub, role):
    exp = dt.datetime.now(dt.timezone.utc) + dt.timedelta(minutes=30)
    return jwt.encode({"sub": sub, "role": role, "exp": exp}, SECRET, algorithm="HS256")

# ---------- dependencies ----------
def get_users():
    return db["users"]

def get_notes():
    return db["notes"]

oauth2 = OAuth2PasswordBearer(tokenUrl="token")

def current_user(token: str = Depends(oauth2)):
    try:
        data = jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError:
        raise HTTPException(401, "Invalid or expired token", headers={"WWW-Authenticate": "Bearer"})
    return {"username": data["sub"], "role": data["role"]}

def require_admin(user=Depends(current_user)):
    if user["role"] != "admin":
        raise HTTPException(403, "Admins only")
    return user

# ---------- models ----------
class UserIn(BaseModel):
    username: str = Field(min_length=3)
    password: str = Field(min_length=6)
    role: str = "member"

class NoteIn(BaseModel):
    title: str = Field(min_length=1)
    body: str = ""

def note_out(doc):
    return {"id": str(doc["_id"]), "title": doc["title"], "body": doc["body"], "owner": doc["owner"]}

# ---------- routes ----------
@app.post("/register", status_code=201, tags=["Auth"])
def register(user: UserIn, users=Depends(get_users)):
    if users.find_one({"username": user.username}):
        raise HTTPException(409, "Username already taken")
    users.insert_one({"username": user.username, "password": hash_password(user.password), "role": user.role})
    return {"username": user.username, "role": user.role}

@app.post("/token", tags=["Auth"])
def login(form: OAuth2PasswordRequestForm = Depends(), users=Depends(get_users)):
    u = users.find_one({"username": form.username})
    if not u or not verify_password(form.password, u["password"]):
        raise HTTPException(401, "Wrong username or password")
    return {"access_token": create_token(u["username"], u["role"]), "token_type": "bearer"}

@app.get("/me", tags=["Auth"])
def me(user=Depends(current_user)):
    return user

@app.post("/notes", status_code=201, tags=["Notes"])
def create_note(note: NoteIn, user=Depends(current_user), notes=Depends(get_notes)):
    doc = {**note.model_dump(), "owner": user["username"]}
    doc["_id"] = notes.insert_one(doc).inserted_id
    return note_out(doc)

@app.get("/notes", tags=["Notes"])
def my_notes(user=Depends(current_user), notes=Depends(get_notes)):
    return [note_out(d) for d in notes.find({"owner": user["username"]})]

@app.delete("/notes/{note_id}", status_code=204, tags=["Notes"])
def delete_note(note_id: str, user=Depends(current_user), notes=Depends(get_notes)):
    if not ObjectId.is_valid(note_id):
        raise HTTPException(404, "Note not found")
    res = notes.delete_one({"_id": ObjectId(note_id), "owner": user["username"]})
    if res.deleted_count == 0:
        raise HTTPException(404, "Note not found")

@app.get("/admin/notes", tags=["Admin"])
def all_notes(admin=Depends(require_admin), notes=Depends(get_notes)):
    return [note_out(d) for d in notes.find()]

In [ ]:
# ===== Option A: Google Colab (active) =====
import importlib, secure_notes
importlib.reload(secure_notes)
from fastapi.testclient import TestClient
client = TestClient(secure_notes.app)

# ===== Option B: VS Code =====
# Terminal: uvicorn secure_notes:app --reload
# Open /docs, use POST /register, then the green Authorize button to log in.

print(client.post("/register", json={"username": "hana", "password": "secret1"}).json())
print(client.post("/register", json={"username": "ravi", "password": "secret2"}).json())
print(client.post("/register", json={"username": "boss", "password": "secret3", "role": "admin"}).json())
print("Duplicate:", client.post("/register", json={"username": "hana", "password": "xxxxxx"}).status_code)

def login(u, p):
    r = client.post("/token", data={"username": u, "password": p})
    return {"Authorization": f"Bearer {r.json()['access_token']}"} if r.status_code == 200 else r.status_code

print("Wrong password:", login("hana", "nope"))
hana, ravi, boss = login("hana", "secret1"), login("ravi", "secret2"), login("boss", "secret3")
print(client.get("/me", headers=hana).json())

In [ ]:
n1 = client.post("/notes", json={"title": "Buy milk"}, headers=hana).json()
client.post("/notes", json={"title": "Read Day 8 post-read"}, headers=hana)
client.post("/notes", json={"title": "Gym at 7"}, headers=ravi)

print("Hana sees:", [n["title"] for n in client.get("/notes", headers=hana).json()])
print("Ravi sees:", [n["title"] for n in client.get("/notes", headers=ravi).json()])
print("No token:", client.get("/notes").status_code)
print("Ravi deletes Hana's note:", client.delete(f"/notes/{n1['id']}", headers=ravi).status_code)
print("Member on admin route:", client.get("/admin/notes", headers=hana).status_code)
print("Admin sees all:", len(client.get("/admin/notes", headers=boss).json()))

## 7. Testing with dependency overrides

In [ ]:
app = secure_notes.app
app.dependency_overrides[secure_notes.current_user] = lambda: {"username": "tester", "role": "admin"}
print(TestClient(app).get("/me").json())        # no token needed in tests
app.dependency_overrides.clear()
print(TestClient(app).get("/me").status_code)   # back to normal: 401

### TODO 1: Add `PATCH /notes/{note_id}` so owners can edit the title or body. Others must get 404.

In [ ]:
# TODO: write your code here

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)